In [ ]:
import sys
sys.path.append("/Users/pablovargas/Documents/grafa")

In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from grafa.client import GrafaClient, GrafaConfig
from dotenv import load_dotenv

load_dotenv()


In [ ]:
import logging
import warnings

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Basic logging setup for Jupyter notebooks
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s - %(name)s - %(message)s',
    force=True  
)

# Set Grafa client logger to show detailed info
grafa_logger = logging.getLogger('grafa.client')
grafa_logger.setLevel(logging.INFO)
# Prevent propagation to avoid duplicates
grafa_logger.propagate = True

# Suppress Neo4j notifications (they're just performance info, not errors)
neo4j_logger = logging.getLogger('neo4j.notifications')
neo4j_logger.setLevel(logging.WARNING)  # Only show warnings and errors, not info


print("✓ Logging configured - duplicates prevented, Neo4j notifications suppressed")

## We define our models

In [ ]:
# Create embedding and LLM objects
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
embedding_dimension = 1536  # or 1024 for some models
llm = ChatOpenAI(model="gpt-5", temperature=0, max_tokens=4096)

### Define config for Grafa

In [ ]:
grafa_config = await GrafaConfig.create(
        embedding_model=embedding_model,
        embedding_dimension=embedding_dimension,
        semantic_similarity_function="cosine",
        llm=llm,
        neo4j_driver=None,  # will be set automatically
        local_storage_path="/tmp/data",     # will be set automatically
    )

### Initialize a new Graph database from a YAML config (FIRST TIME)

In [ ]:
client = await GrafaClient.from_yaml(
        yaml_path="/Users/pablovargas/Documents/grafa/schema.yaml",
        db_name="my_db5",  # This will create the database in Neo4j
        grafa_config=grafa_config,
    )

### Connect to Existing Database (SUBSEQUENT TIMES)

Once created a database using `from_yaml()`, you can connect to it using `create()`.

In [ ]:
client = await GrafaClient.create(
    db_name="my_db",  
    grafa_config=grafa_config
)

### Ingesting

In [ ]:
document, chunks, entities, relationships = await client.ingest_file(
    document_name="mercadolibre",
    document_path="/Users/pablovargas/Documents/grafa/example.txt",  
    context="Business analysis document",
    author="Pablo"
)

In [ ]:
relationships = client.get_user_defined_relationship_types()
for rel in relationships:
    print(rel)

### Step to step ingesting

In [ ]:
# Step 1: Upload document
file = await client.upload_file(
    document_name="my_document",
    document_path="path/to/document.txt"
)

# Step 2: Process/transcribe the file
processed_file = await client.process_file(file)

# Step 3: Chunk the document
document = processed_file._grafa_document
chunks = await client.chunk_document(document, max_token_chunk_size=500)

# Step 4: Extract entities and relationships from each chunk
for chunk in chunks:
    entities, relationships = await client.process_chunk(chunk)

### Query

In [ ]:
results = await client.knowledgebase_query("Quien es Marcos Galperin?")
print(results)

In [ ]:
# More advanced search
search_results = await client.similarity_search(
    query="Quien es Marcos Galperin?",
    node_types=["Concept", "Person"], 
    limit=10,
    search_mode="hybrid" 
)
print(search_results)

In [ ]:
node_types = client.get_user_defined_node_types()
print("Available node types:", [nt.__name__ for nt in node_types])

relationships = client.get_user_defined_relationship_types()
print("Available relationships:", [r.type for r in relationships])

# Search and explore
concepts = await client.similarity_search("innovation", node_types=["Concept"])
for concept in concepts:
    print(f"Found: {concept['name']} (score: {concept.get('semantic_score', 'N/A')})")